# vLLM 大语言模型推理教程

本教程详细介绍 vLLM 大语言模型推理引擎的使用，包括：

1. **vLLM 基础**: PagedAttention 和 Continuous Batching
2. **模型加载**: 配置和量化
3. **文本生成**: 采样参数和批量推理
4. **高级功能**: LoRA 和投机解码

---

## 什么是 vLLM？

vLLM 是专门为大语言模型 (LLM) 设计的高性能推理引擎：

```
传统 LLM 推理:              vLLM 推理:
┌─────────────────┐        ┌─────────────────┐
│  每个请求独立    │        │  PagedAttention │
│  KV Cache       │        │  共享 KV Cache  │
│                 │        │                 │
│  内存浪费严重    │   →    │  内存利用率高   │
│  吞吐量低       │        │  吞吐量高       │
└─────────────────┘        └─────────────────┘
```

**注意**: vLLM 需要 GPU 支持

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np

# 检查 vLLM
try:
    from vllm import LLM, SamplingParams
    print("vLLM 已安装")
    VLLM_AVAILABLE = True
except ImportError:
    print("vLLM 未安装")
    print("安装命令: pip install vllm")
    VLLM_AVAILABLE = False

# 检查 PyTorch 和 CUDA
try:
    import torch
    print(f"PyTorch 版本: {torch.__version__}")
    print(f"CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU 内存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except ImportError:
    print("PyTorch 未安装")

## 1. PagedAttention 原理

### 传统 KV Cache 问题

```
请求 1: [████████████░░░░░░░░]  预分配但未使用
请求 2: [██████░░░░░░░░░░░░░░]  大量内存浪费
请求 3: [████████████████░░░░]
        ↑ 实际使用    ↑ 浪费
```

### PagedAttention 解决方案

```
物理内存块 (Pages):
┌────┬────┬────┬────┬────┬────┬────┬────┐
│ P0 │ P1 │ P2 │ P3 │ P4 │ P5 │ P6 │ P7 │
└────┴────┴────┴────┴────┴────┴────┴────┘
  ↑    ↑    ↑    ↑    ↑    ↑
  │    │    │    │    │    │
请求1: [P0, P1, P2]
请求2: [P3, P4]
请求3: [P5, P6, P7]

- 按需分配，无浪费
- 支持动态增长
- 内存利用率接近 100%
```

## 2. 使用封装的模块

In [ ]:
from vllm_inference import (
    SamplingConfig,
    EngineConfig,
    GenerationOutput,
    QuantizationMethod,
    VLLM_AVAILABLE
)

# 采样配置
sampling_config = SamplingConfig(
    temperature=0.8,
    top_p=0.95,
    top_k=50,
    max_tokens=256,
    presence_penalty=0.1,
    frequency_penalty=0.1
)

print("采样配置:")
print(f"  温度: {sampling_config.temperature}")
print(f"  Top-p: {sampling_config.top_p}")
print(f"  Top-k: {sampling_config.top_k}")
print(f"  最大 tokens: {sampling_config.max_tokens}")

In [ ]:
# 引擎配置
engine_config = EngineConfig(
    model="meta-llama/Llama-2-7b-hf",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.9,
    max_model_len=4096,
    dtype="float16"
)

print("引擎配置:")
print(f"  模型: {engine_config.model}")
print(f"  张量并行: {engine_config.tensor_parallel_size}")
print(f"  GPU 内存利用率: {engine_config.gpu_memory_utilization}")
print(f"  最大序列长度: {engine_config.max_model_len}")
print(f"  数据类型: {engine_config.dtype}")

## 3. 采样参数详解

In [ ]:
# 不同采样策略
sampling_strategies = {
    "贪婪解码": SamplingConfig(temperature=0.0, top_p=1.0, top_k=-1),
    "低温采样": SamplingConfig(temperature=0.3, top_p=0.9),
    "标准采样": SamplingConfig(temperature=0.7, top_p=0.95),
    "高温采样": SamplingConfig(temperature=1.2, top_p=0.95),
    "Top-k 采样": SamplingConfig(temperature=0.8, top_k=50),
    "Nucleus 采样": SamplingConfig(temperature=0.8, top_p=0.9),
}

print("采样策略对比:")
print(f"{'策略':<15} {'温度':<8} {'Top-p':<8} {'Top-k':<8} {'特点'}")
print("-" * 60)

strategy_features = {
    "贪婪解码": "确定性，最高概率",
    "低温采样": "保守，较少随机性",
    "标准采样": "平衡创造性和连贯性",
    "高温采样": "高创造性，可能不连贯",
    "Top-k 采样": "限制候选词数量",
    "Nucleus 采样": "动态候选词集合",
}

for name, config in sampling_strategies.items():
    print(f"{name:<15} {config.temperature:<8} {config.top_p:<8} {config.top_k:<8} {strategy_features[name]}")

## 4. 量化配置

In [ ]:
# 量化方法
print("支持的量化方法:")
for method in QuantizationMethod:
    print(f"  - {method.name}: {method.value}")

# 量化配置示例
quantization_configs = {
    "无量化 (FP16)": EngineConfig(
        model="meta-llama/Llama-2-7b-hf",
        quantization=QuantizationMethod.NONE,
        dtype="float16"
    ),
    "GPTQ 量化": EngineConfig(
        model="TheBloke/Llama-2-7B-GPTQ",
        quantization=QuantizationMethod.GPTQ
    ),
    "AWQ 量化": EngineConfig(
        model="TheBloke/Llama-2-7B-AWQ",
        quantization=QuantizationMethod.AWQ
    ),
}

print("\n量化配置对比:")
print(f"{'配置':<20} {'模型':<35} {'量化方法'}")
print("-" * 70)
for name, config in quantization_configs.items():
    quant = config.quantization.value if config.quantization.value else "None"
    print(f"{name:<20} {config.model:<35} {quant}")

## 5. vLLM 基本使用 (需要 GPU)

以下代码需要 vLLM 和 GPU 才能运行。

In [ ]:
if VLLM_AVAILABLE:
    # 使用小模型进行演示
    print("加载模型 (这可能需要几分钟)...")
    
    try:
        llm = LLM(
            model="facebook/opt-125m",  # 小模型用于演示
            gpu_memory_utilization=0.5,
            max_model_len=512
        )
        print("模型加载成功!")
        MODEL_LOADED = True
    except Exception as e:
        print(f"模型加载失败: {e}")
        MODEL_LOADED = False
else:
    print("跳过 vLLM 示例 (vLLM 未安装)")
    MODEL_LOADED = False

In [ ]:
if VLLM_AVAILABLE and MODEL_LOADED:
    # 基本文本生成
    prompts = [
        "The future of artificial intelligence is",
        "Machine learning can help us",
        "Deep learning models are"
    ]
    
    sampling_params = SamplingParams(
        temperature=0.8,
        top_p=0.95,
        max_tokens=50
    )
    
    print("生成文本...")
    outputs = llm.generate(prompts, sampling_params)
    
    print("\n生成结果:")
    print("=" * 60)
    for output in outputs:
        print(f"提示: {output.prompt}")
        print(f"生成: {output.outputs[0].text}")
        print("-" * 60)
else:
    print("跳过文本生成示例")

## 6. 批量推理性能

In [ ]:
import time

if VLLM_AVAILABLE and MODEL_LOADED:
    # 测试不同批次大小的性能
    batch_sizes = [1, 4, 8, 16]
    results = []
    
    base_prompt = "The quick brown fox jumps over the lazy dog. "
    
    for batch_size in batch_sizes:
        prompts = [base_prompt] * batch_size
        
        sampling_params = SamplingParams(
            temperature=0.8,
            max_tokens=32
        )
        
        # 预热
        _ = llm.generate(prompts[:1], sampling_params)
        
        # 计时
        start = time.perf_counter()
        outputs = llm.generate(prompts, sampling_params)
        elapsed = time.perf_counter() - start
        
        # 统计 tokens
        total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        
        results.append({
            'batch_size': batch_size,
            'time_s': elapsed,
            'total_tokens': total_tokens,
            'tokens_per_sec': total_tokens / elapsed,
            'latency_per_request': elapsed / batch_size * 1000
        })
    
    print("批量推理性能:")
    print(f"{'Batch':<8} {'Time(s)':<10} {'Tokens':<10} {'Tok/s':<12} {'Latency(ms)'}")
    print("-" * 55)
    for r in results:
        print(f"{r['batch_size']:<8} {r['time_s']:<10.3f} {r['total_tokens']:<10} {r['tokens_per_sec']:<12.1f} {r['latency_per_request']:.1f}")
else:
    print("跳过批量推理性能测试")

## 7. Continuous Batching

```
传统静态批处理:
时间 →
请求1: [████████████████████████████████]
请求2: [████████░░░░░░░░░░░░░░░░░░░░░░░░] 等待
请求3: [████████████████░░░░░░░░░░░░░░░░] 等待
       ↑ 必须等最长请求完成才能处理新请求

Continuous Batching:
时间 →
请求1: [████████████████████████████████]
请求2: [████████]→完成→[新请求5开始...]
请求3: [████████████████]→完成→[新请求6...]
请求4:     [████████████████████████]
       ↑ 请求完成后立即处理新请求
```

## 8. LoRA 适配器

In [ ]:
# LoRA 配置示例
lora_config = EngineConfig(
    model="meta-llama/Llama-2-7b-hf",
    enable_lora=True,
    max_loras=4,
    max_lora_rank=16
)

print("LoRA 配置:")
print(f"  启用 LoRA: {lora_config.enable_lora}")
print(f"  最大 LoRA 数量: {lora_config.max_loras}")
print(f"  最大 LoRA 秩: {lora_config.max_lora_rank}")

print("\nLoRA 使用示例 (伪代码):")
print("""
from vllm.lora.request import LoRARequest

# 创建 LoRA 请求
lora_request = LoRARequest(
    "adapter_name",
    1,  # adapter_id
    "/path/to/lora/adapter"
)

# 使用 LoRA 生成
outputs = llm.generate(
    prompts,
    sampling_params,
    lora_request=lora_request
)
""")

## 9. 投机解码 (Speculative Decoding)

In [ ]:
# 投机解码配置
speculative_config = EngineConfig(
    model="meta-llama/Llama-2-70b-hf",
    speculative_model="meta-llama/Llama-2-7b-hf",
    num_speculative_tokens=5
)

print("投机解码配置:")
print(f"  目标模型: {speculative_config.model}")
print(f"  草稿模型: {speculative_config.speculative_model}")
print(f"  投机 tokens: {speculative_config.num_speculative_tokens}")

print("\n投机解码原理:")
print("""
1. 小模型 (草稿模型) 快速生成 N 个候选 tokens
2. 大模型 (目标模型) 并行验证这 N 个 tokens
3. 接受正确的 tokens，拒绝错误的
4. 从第一个错误位置重新生成

优势:
- 减少大模型的前向传播次数
- 保持与大模型相同的输出质量
- 可获得 2-3x 加速
""")

## 10. API 服务模式

In [ ]:
print("启动 vLLM API 服务器:")
print("""
# 命令行启动
python -m vllm.entrypoints.openai.api_server \\
    --model meta-llama/Llama-2-7b-hf \\
    --port 8000 \\
    --tensor-parallel-size 1

# 客户端调用 (兼容 OpenAI API)
import openai

client = openai.OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy"
)

response = client.chat.completions.create(
    model="meta-llama/Llama-2-7b-hf",
    messages=[
        {"role": "user", "content": "Hello!"}
    ]
)

print(response.choices[0].message.content)
""")

## 总结

本教程介绍了 vLLM 的核心功能：

1. **PagedAttention**: 高效内存管理
2. **Continuous Batching**: 动态批处理
3. **采样参数**: 控制生成质量
4. **量化支持**: GPTQ/AWQ
5. **高级功能**: LoRA、投机解码

### 最佳实践

- 使用量化模型减少内存占用
- 调整 `gpu_memory_utilization` 平衡内存和吞吐
- 使用 Continuous Batching 提高吞吐量
- 考虑投机解码加速长文本生成